# MENA Drought Early Warning Forecasting

**Objective:** forecast drought severity 1 and 3 months ahead using real Earth Engine data.  
**Core datasets:** MODIS NDVI, CHIRPS rainfall, MODIS land surface temperature, TerraClimate.  
**Validation design:** temporal train/validation/test split.

---

This notebook turns the first drought-mapping project into an early-warning workflow.


## Notebook Roadmap

1. Environment setup  
2. Earth Engine and grid setup  
3. Monthly extraction to a pandas table  
4. Feature engineering and future targets  
5. Forecasting experiments  
6. Export tables, figures, and a map


In [ ]:
# Run once if needed
# !pip install earthengine-api geemap folium pandas numpy matplotlib scikit-learn


## 1. Environment Setup

**What**
- Load the reusable code from `src/`
- Create output folders
- Prepare the shared project config

**How**
- Add `src/` to `sys.path`
- Import helpers for extraction, features, modeling, and reporting

**Why**
- This keeps the notebook readable and the workflow reusable


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mena_drought_early_warning.config import ProjectConfig
from mena_drought_early_warning.utils import ensure_project_directories
from mena_drought_early_warning.gee_extract import (
    initialize_earth_engine,
    build_mena_bbox,
    build_fishnet,
    extract_monthly_grid_dataframe,
)
from mena_drought_early_warning.features import (
    add_forecast_features,
    default_feature_columns,
    build_forecast_dataset,
)
from mena_drought_early_warning.modeling import (
    temporal_split,
    fit_random_forest,
    fit_xgboost_classifier,
    xgboost_available,
    predict_persistence,
    evaluate_predictions,
    classification_report_text,
    feature_importance_series,
)
from mena_drought_early_warning.sequences import build_sequence_dataset
from mena_drought_early_warning.deep_learning import tensorflow_available, lstm_architecture_note
from mena_drought_early_warning.reporting import (
    save_confusion_matrix,
    save_feature_importance_plot,
    save_metric_comparison,
    export_forecast_table,
    create_forecast_map,
)

config = ProjectConfig()
ensure_project_directories(config)

print(f"Project root: {config.project_root}")
print(f"Figures directory: {config.figures_dir}")
print(f"XGBoost available: {xgboost_available()}")
print(f"TensorFlow available: {tensorflow_available()}")


## 2. Earth Engine and Spatial Configuration

**What**
- Initialize Earth Engine
- Build the MENA bounding box and fishnet grid

**How**
- Use the shared config to define geography and grid size

**Why**
- Forecasting needs stable spatial units so monthly raster data can become model rows


In [ ]:
# If needed, use initialize_earth_engine(authenticate=True) once.
initialize_earth_engine()

mena_bbox = build_mena_bbox(config)
grid_fc = build_fishnet(config)

print("Earth Engine initialized.")
print("Grid cells:", grid_fc.size().getInfo())


## 3. Monthly Extraction to a pandas Table

**What**
- Extract monthly real-world variables over the grid
- Convert the result to pandas

**How**
- Use MODIS NDVI, CHIRPS rainfall, MODIS LST, and TerraClimate
- Extract the table month by month so the workflow is more realistic and memory-safe

**Why**
- This creates the cell-month table required for time-series forecasting


In [ ]:
raw_df = extract_monthly_grid_dataframe(config, grid_fc, mena_bbox)

print("Monthly grid-cell table shape:", raw_df.shape)
raw_df.head()


## 4. Feature Engineering and Future Targets

**What**
- Build anomalies, rolling features, and drought classes
- Shift the class into the future for forecast targets

**How**
- Compute climatologies by cell and calendar month
- Build `t+1` and `t+3` targets from future drought class values

**Why**
- This changes the problem from monitoring current drought to forecasting future drought


In [ ]:
feature_df = add_forecast_features(raw_df, config)
FEATURE_COLUMNS = default_feature_columns()

print("Feature table shape:", feature_df.shape)
print("Model features:", FEATURE_COLUMNS)

feature_df[["date", "cell_id", "ndvi", "ndvi_anomaly", "rainfall", "pdsi", "drought_label"]].head()


## 5. Forecasting Experiments

**What**
- Compare persistence and Random Forest for horizons `t+1` and `t+3`
- Prepare the same workflow for an optional XGBoost benchmark

**How**
- Build a dataset per horizon
- Split data through time
- Fit Random Forest on training years only
- Fit XGBoost on the same split if the package is available

**Why**
- Temporal evaluation is more honest than a random split for forecasting tasks


In [ ]:
results = []
test_predictions = {}

for horizon in config.forecast_horizons:
    model_df, target_column = build_forecast_dataset(feature_df, horizon, FEATURE_COLUMNS, config)
    split = temporal_split(model_df, config.train_end_date, config.valid_end_date)

    rf_model = fit_random_forest(split.train, FEATURE_COLUMNS, target_column)
    rf_test_pred = rf_model.predict(split.test[FEATURE_COLUMNS])
    persistence_pred = predict_persistence(split.test)

    rf_metrics = evaluate_predictions(split.test[target_column], rf_test_pred)
    persistence_metrics = evaluate_predictions(split.test[target_column], persistence_pred)

    print(f"\n===== Horizon t+{horizon} =====")
    print(classification_report_text(split.test[target_column], rf_test_pred, sorted(config.class_names), [config.class_names[i] for i in sorted(config.class_names)]))

    save_confusion_matrix(
        split.test[target_column],
        rf_test_pred,
        config.class_names,
        f"Random Forest Confusion Matrix - Horizon t+{horizon}",
        config.figures_dir / f"rf_confusion_matrix_h{horizon}.png",
    )
    save_confusion_matrix(
        split.test[target_column],
        persistence_pred,
        config.class_names,
        f"Persistence Confusion Matrix - Horizon t+{horizon}",
        config.figures_dir / f"persistence_confusion_matrix_h{horizon}.png",
    )

    importance = feature_importance_series(rf_model, FEATURE_COLUMNS)
    save_feature_importance_plot(
        importance,
        f"Random Forest Feature Importance - Horizon t+{horizon}",
        config.figures_dir / f"rf_feature_importance_h{horizon}.png",
    )

    results.append({"horizon": horizon, "model": "Persistence", **persistence_metrics})
    results.append({"horizon": horizon, "model": "Random Forest", **rf_metrics})

    if xgboost_available():
        xgb_model = fit_xgboost_classifier(split.train, split.valid, FEATURE_COLUMNS, target_column)
        xgb_test_pred = xgb_model.predict(split.test[FEATURE_COLUMNS])
        xgb_metrics = evaluate_predictions(split.test[target_column], xgb_test_pred)
        results.append({"horizon": horizon, "model": "XGBoost", **xgb_metrics})

    horizon_test_df = split.test.copy()
    horizon_test_df["rf_prediction"] = rf_test_pred
    horizon_test_df["rf_prediction_label"] = horizon_test_df["rf_prediction"].map(config.class_names)
    horizon_test_df["persistence_prediction"] = persistence_pred
    horizon_test_df["persistence_prediction_label"] = horizon_test_df["persistence_prediction"].map(config.class_names)
    test_predictions[horizon] = horizon_test_df

metrics_df = pd.DataFrame(results)
metrics_df.to_csv(config.tables_dir / "forecast_model_metrics.csv", index=False)

for horizon in config.forecast_horizons:
    metric_subset = metrics_df[metrics_df["horizon"] == horizon].set_index("model")[["accuracy", "balanced_accuracy", "macro_f1"]]
    save_metric_comparison(
        metric_subset,
        f"Forecast Metric Comparison - Horizon t+{horizon}",
        config.figures_dir / f"forecast_metric_comparison_h{horizon}.png",
    )

metrics_df


## 6. Sequence Modeling Preparation

**What**
- Prepare a sequence dataset for a future LSTM experiment

**How**
- Build rolling 6-month sequences from the same forecast table
- Keep the same target design and temporal evaluation logic

**Why**
- This makes the deep-learning extension structured and comparable rather than improvised


In [ ]:
sequence_horizon = 1
sequence_df, sequence_target_column = build_forecast_dataset(feature_df, sequence_horizon, FEATURE_COLUMNS, config)
X_seq, y_seq, meta_seq = build_sequence_dataset(
    sequence_df,
    feature_columns=FEATURE_COLUMNS,
    target_column=sequence_target_column,
    sequence_length=6,
)

print("Sequence tensor shape:", X_seq.shape)
print("Sequence target shape:", y_seq.shape)
print("TensorFlow available:", tensorflow_available())
print("LSTM architecture note:")
print(lstm_architecture_note())
meta_seq.head()


## 7. Export Tables, Figures, and a Map

**What**
- Export a selected forecast issue month and map it

**How**
- Filter the test forecasts to a selected issue month
- Save a CSV and HTML map

**Why**
- This creates map-ready early-warning outputs


In [ ]:
selected_horizon = 1
forecast_df = test_predictions[selected_horizon].copy()
issue_date = pd.Timestamp(config.forecast_issue_date)

selected_issue_df = forecast_df[forecast_df["date"] == issue_date].copy()
if selected_issue_df.empty:
    issue_date = forecast_df["date"].max()
    selected_issue_df = forecast_df[forecast_df["date"] == issue_date].copy()

csv_path = config.tables_dir / f"forecast_issue_table_h{selected_horizon}_{issue_date.strftime('%Y_%m')}.csv"
map_path = config.maps_dir / f"forecast_map_h{selected_horizon}_{issue_date.strftime('%Y_%m')}.html"

export_columns = [
    "cell_id", "date", "lat", "lon", "ndvi", "rainfall", "lst_c", "pdsi", "vpd",
    "drought_label", "rf_prediction_label", "persistence_prediction_label"
]

export_forecast_table(selected_issue_df, csv_path, export_columns)
forecast_map = create_forecast_map(
    selected_issue_df,
    predicted_label_column="rf_prediction_label",
    save_path=map_path,
    config=config,
    title_prefix=f"Random Forest Forecast t+{selected_horizon}",
)

print(f"Exported forecast table: {csv_path}")
print(f"Exported forecast map: {map_path}")
forecast_map


## Final Interpretation

This notebook demonstrates a clean early-warning baseline:

- real data
- future targets
- temporal validation
- baseline benchmarking
- exportable forecast outputs

Possible extensions:

- compare Random Forest and XGBoost on the same temporal split
- train an LSTM using the prepared 6-month sequences if TensorFlow is available
- use grouped spatial validation
- replace the fishnet with better boundaries
- compare proxy labels against external drought event data


## Model Comparison Checklist

Model sequence:

1. Run `Persistence` first.
2. Run `Random Forest` on the same horizon and temporal split.
3. Run `XGBoost` only if the package is installed.
4. Run `LSTM` only after the tabular models are stable.

Keep these fixed across all models:

- target definition
- forecast horizon
- train / validation / test periods
- evaluation metrics

Metrics:

- accuracy
- balanced accuracy
- macro F1

Model selection note:

- feature the simplest model unless a more complex model wins clearly and consistently
